# Autograd

Automatic differentiation (or commonly referred to as Autograd) is one of the central components that allow for modern machine learning algorithms to work in the present day.

Autograd's entire purpose in life is to do one thing. Compute derivatives on a computer. You might think, isn't that easy? Just approximate the following:

$$\frac{df}{dx} \sim \frac{f(x+dx)-f(x)}{dx}$$

But no this is numerically unstable. When on a computer you divide by a really small number, the order of operations can cause things to blow up. That sucks :( 

How can things be better? Firstly, when you know the function you want to compute the derivative to, why can't you just... get the exact derivative?

$$f(x) = \sin(x+e^x)$$
I can easily take the derivative of that 
$$f'(x) = \cos(x+e^x)(1+e^x)$$
Done! We can easily type $f'$ on a compute and get the exact derivative of $f$. The question is can we do so **automatically**?

## Computational Graph

So we know the rules already from calculus. Product rule, chain rule etc. We can just hardcode those rules on a computer! What is important however is keeping track of everything. Why do we need to keep track?

Well we know how to manually compute a few derivatives like $\sin,\cos, \log, x^2$ etc. But for any general function  like $\sin(x+e^x)$ or whatever, we don't want to memorize that but automatically build up from our knowledge of derivatives of elementary functions like $\sin,\cos,\log$ etc.  To build up to that derivative, we applied **chain rule**. The chain rule requires us to keep track of how these elementary functions are nested, combined and composed. This is what a **computational graph** is for.

A computation graph, is a DAG (directed, acyclic graph). A graph has edges (connections) and nodes. Lets see how this represents a very simple function.

$$f(x,y,z) = (x + y) \cdot z$$
If we have the function $f = (x + y) \cdot z$, the system creates nodes for variables ($x, y, z$) and operations ($+, \cdot$).

```mermaid
graph LR
    %% Inputs
    x((x))
    y((y))
    z((z))
    
    %% Operations
    add((+))
    mul(( * ))
    
    %% Output
    f((f))

    %% Connections
    x -->| | add
    y -->| | add
    add -->|q = x+y| mul
    z -->| | mul
    mul -->| | f
    
    %% Styling
    classDef input fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef op fill:#fff3e0,stroke:#ff9800,stroke-width:2px;
    class x,y,z input;
    class add,mul op;
```

Why would this be a good way of storing the function? Well... it lets us easily traverse it to compute the derivatives!

Consider computing the derivative of $f$ in respect to $x$. We start at the end $f$ we move up the graph to the left and we get a product. We know a produce rule is $g'h+h'g$ right? so we go to the left and the right to get our $g,h$ values we see we have $z$ which is itself a variable/elementary (cannot be broken down more). The derivative of this in respect to $x$ is $0$. So $g =z, g'=0$ we can fill in the values $g'h+h'g = (0)h+h'z$. All that is left is to find $h, h'$ we know that $h=x+y$ and we see that $h'=1$ in respect to $x$ and so to build the final result we $f' = z$.

We got a taste of this, now lets build it!

#### Custom Data Structure

To make this work we need to define a **new** way of storing a number. 

What we are going to do is **overwrite** python's definition of *add* and *multiply*

```
def __add__ (self, stuff whatever):
    blah blah blah
```
Lets you redefine what addition means!



In [18]:
import math

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self._prev = set(_children) # here is where you connect previous children
        self._op = _op  # Storing the operation name helps for debugging

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        return out
        
    def log(self):
        out = Value(math.log(other.data), (self), 'log')
        return out

In [17]:
# 1. Initialize input leaves (no operations attached)
x = Value(1)
y = Value(2)

# 2. The Forward Pass!
# This implicitly calls x.__add__(y)
add = x + y 

# 3. View the graph structure
print("Result Node: ", add)
print("Result Data: ", add.data)
print("Result Op:   ", add._op)
print("Result Prev: ", add._prev)

Result Node:  <__main__.Value object at 0x1122d6c10>
Result Data:  3
Result Op:    +
Result Prev:  {<__main__.Value object at 0x1122d6e90>, <__main__.Value object at 0x1122a74d0>}


### Step 1 Forward Pass

Now lets say we want to implement the same idea but with the functions $\sin$ and $e^x$. Can you implement the forward pass for those?

Fill out the missing $exp, \sin$

In [ ]:
import math

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0          # The gradient starts at 0
        self._backward = lambda: None # Function to compute gradients for children
        self._prev = set(_children)   # The nodes that created this node

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # --- Addition (+) ---
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        return out

    # --- Multiplication (*) ---
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        return out

    def log(self):
        out = Value(math.log(other.data), (self), 'log')
        return out

    # --- Sine (sin) ---
    def sin(self):
        ## TODO (hint) see the log function above for hints

        ##
        return out

    # --- Exponential (e^x) ---
    def exp(self):
        ### TODO

        return out



### Testing Forward Pass

Lets see how this rolls!

In [21]:
# 1. Define inputs
x = Value(2.0)
y = Value(3.0)

# 2. Forward Pass: Build the graph
# f = sin(x) + (e^x * y)
term1 = x.sin()
term2 = x.exp() * y
f = term1 + term2

print("--- After Forward Pass ---")
print(f"f: {f}") # Should be sin(2) + e^2 * 3 = 0.9093 + 7.3890 * 3 = 23.076

NameError: name 'out' is not defined

## Derivatives

Alright now since we know the derivatives of each individual elementary function, how about we start writing those?

I have implemented ones for $+$ and $\log$. Implement the rest!

In [20]:
import math

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0          # The gradient starts at 0
        self._backward = lambda: None # Function to compute gradients for children
        self._prev = set(_children)   # The nodes that created this node

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # --- Addition (+) ---
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # Local derivative of addition is 1.0
            # Chain rule: 1.0 * out.grad 
            self.grad += 1.0 * out.grad # update yourself
            other.grad += 1.0 * out.grad # update your friend!
            
        out._backward = _backward # set this backward function as the key backward component
        
        return out

    # --- Multiplication (*) ---
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        ### TODO Implement backward here (hint.. see add and remember product rule!)

        
        return out

    def log(self):
        out = Value(math.log(other.data), (self), 'log')

        def _backward():
            # derivative of log is 1/x but always remember to chain rule so you multiply
            # by out.grad at the end
            self.grad = (1/out.data)* out.grad 
            
        out._backward = _backward
        return out

    # --- Sine (sin) ---
    def sin(self):
        ## TODO (hint) see the log function above for hints

        ### TODO Implement backward here 

        
        return out

    # --- Exponential (e^x) ---
    def exp(self):
        ### TODO

        ### TODO Implement backward here 

        
        return out

## Autograd Engine (No task just read the implementation)

Lets actually write out the engine of this and get you your beautifully differentiated function. Let me describe to you the strategy.

### Step 1
This section builds a linear ordering of the computational graph. So in graph theory, the computational graph is a Directed Acyclic Graph (DAG) as mentioned. A topological sort, all it does is it arranges the nodes so that every directed edge goes from a node earlier in the sequence to a node later in the sequence. This is called a Depth First Search or "post-order traversal". 

What it does is it goes to through each child node, if it is not the base node it 

### The Step-by-Step Trace

Let's trace exactly how the array builds up for our function $f = \sin(x) + e^x \cdot y$. 

Remember, `build_topo` starts at the very end of the graph (the `add` node) and works backward, asking: *"What did I need to compute this?"*

1. **Start at `add`:** * It is marked as `visited`.
   * It checks its prerequisites: `sin` and `mult`.
   * The loop pauses `add` and calls `build_topo(sin)`.
2. **Inside `sin`:**
   * It is marked as `visited`.
   * It checks its prerequisite: `x`.
   * The loop pauses `sin` and calls `build_topo(x)`.
3. **Inside `x`:**
   * It is marked as `visited`.
   * It has *no* prerequisites (it is a raw input). The `for` loop skips.
   * We hit the end of the function! `x` is appended to the list.
   * **`topo` = `[x]`**
4. **Back to `sin`:**
   * `sin` has finished its loop. We hit the end of its function.
   * **`topo` = `[x, sin]`**
5. **Back to `add`:**
   * `add` finished looking at `sin`. Now its loop moves to the next prerequisite: `mult`.
   * It calls `build_topo(mult)`.
6. **Inside `mult`:**
   * It is marked as `visited`.
   * It checks its prerequisites: `exp` and `y`.
   * It calls `build_topo(exp)`.
7. **Inside `exp`:**
   * Marked as `visited`.
   * Prerequisite is `x`. It calls `build_topo(x)`.
   * **Wait!** `x` is already in the `visited` set. The `if v not in visited:` check fails. It immediately returns.
   * `exp` finishes its loop and gets appended.
   * **`topo` = `[x, sin, exp]`**
8. **Back to `mult`:**
   * It calls `build_topo(y)`.
   * `y` has no prerequisites. Loop skips, `y` gets appended.
   * **`topo` = `[x, sin, exp, y]`**
   * `mult` finishes its loop and gets appended.
   * **`topo` = `[x, sin, exp, y, mult]`**
9. **Finally, back to `add`:**
   * `add` has now finished checking all its prerequisites.
   * It gets appended.
   * **`topo` = `[x, sin, exp, y, mult, add]`**

Notice how the `add` node was the very first one we looked at, but the very last one to be appended. Because the `backward()` loop later reads this list in `reversed()` order, it processes `add` first, passing gradients down exactly in the reverse order they were computed.


## Step 2 Base case

For any recursive call (if you've ever taken a computer science class) you would remember that eventually you need to stop! This is where you set the gradient to be one. 

Remember by default, every Value object is initialized with a grad of 0.0. If we start the chain rule multiplying by $0$, every gradient in the entire network will remain $0$. We must "seed" the gradient at the very end of the graph. Mathematically, the derivative of a variable with respect to itself is always $1$ ($\frac{\partial f}{\partial f} = 1$).We set the final output node's gradient to 1.0 so the chain rule has a starting value to multiply against.


## Step 3 

Now that we have our nodes perfectly sorted and our base gradient seeded, we execute the math.

reversed(topo): Because topo ends with the final output node, reversing it means we start at the output and walk strictly backwards toward the inputs.

node._backward(): As we visit each node in this reversed order, we call its specific _backward() function.

If the node was an addition, its _backward() function takes its current .grad and passes it equally to its parents.

If it was a multiplication, it applies the product rule to pass the gradients backward.

Because of the topological sort, we have a mathematical guarantee: by the time the for loop reaches a specific node to call its _backward() function, that node has already received the fully accumulated gradient from every single node that depended on it. Without the topological sort, the engine might try to execute _backward() on a node before its total gradient was fully calculated, resulting in mathematically incorrect derivatives.

In [ ]:
import math

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0          # The gradient starts at 0
        self._backward = lambda: None # Function to compute gradients for children
        self._prev = set(_children)   # The nodes that created this node

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # --- Addition (+) ---
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # Local derivative of addition is 1.0
            # Chain rule: 1.0 * out.grad 
            self.grad += 1.0 * out.grad # update yourself
            other.grad += 1.0 * out.grad # update your friend!
            
        out._backward = _backward # set this backward function as the key backward component
        
        return out

    # --- Multiplication (*) ---
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        ### TODO Implement backward here (hint.. see add and remember product rule!)

        
        return out

    def log(self):
        out = Value(math.log(other.data), (self), 'log')

        def _backward():
            # derivative of log is 1/x but always remember to chain rule so you multiply
            # by out.grad at the end
            self.grad = (1/out.data)* out.grad 
            
        out._backward = _backward
        return out

    # --- Sine (sin) ---
    def sin(self):
        ## TODO (hint) see the log function above for hints

        ### TODO Implement backward here 

        
        return out

    # --- Exponential (e^x) ---
    def exp(self):
        ### TODO

        ### TODO Implement backward here 

        
        return out

    def backward(self):
        # 1. Topologically sort the graph to ensure we go backwards in the correct order
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # 2. Set the base gradient to 1.0 (dL / dL = 1)
        self.grad = 1.0
        
        # 3. Apply the chain rule backwards through the sorted nodes
        for node in reversed(topo):
            node._backward()

# Test it out!


In [ ]:
# 1. Initialize variables
x = Value(0.5)
y = Value(1.2)

# 2. Build the computational graph
a = x.sin()
b = y.exp()
f = a * b

# 3. Run the backward pass
f.backward()

# 4. View the results
print(f"f data: {f.data:.4f}")       # Forward pass result
print(f"x grad (df/dx): {x.grad:.4f}") # Derivative wrt x
print(f"y grad (df/dy): {y.grad:.4f}") # Derivative wrt y